## Edge AI Keyword Spotting: Data Preparation and Model Training

## Import Libraries and Initialize Random Seeds:

In [1]:
import pathlib
import librosa
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random
import wave
import os
import soundfile as sf
from tqdm import tqdm

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


## Download and Extract the Speech Commands Dataset

In [2]:
# Google Speech Commands Dataset URL
DATASET_URL = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"

# Expected path after extraction
data_dir = pathlib.Path("./data/speech_commands")

if not data_dir.exists():
    data_dir = pathlib.Path(
        tf.keras.utils.get_file(
            "speech_commands",
            origin=DATASET_URL,
            untar=True,
            cache_dir=".",
            cache_subdir="data"
        )
    )
    print("Downloaded dataset:", data_dir)
else:
    print("Dataset already exists:", data_dir)

2428923189/2428923189 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step
Downloaded dataset: data/speech_commands


## Generate Silence Samples for the Dataset

In [3]:
# Configuration
BACKGROUND_DIR = "data/speech_commands/_background_noise_"
MY_SILENCE_DIR = "my_silence"
OUTPUT_DIR = "data/speech_commands/silence"

TARGET_SR = 16000
CLIP_LENGTH = TARGET_SR

NUM_SAMPLES = 4000

MIN_GAIN = 0.05
MAX_GAIN = 0.30

PURE_SILENCE_RATIO = 0.05
GOOGLE_RATIO = 0.45
ROOM_RATIO = 0.50

assert abs(PURE_SILENCE_RATIO + GOOGLE_RATIO + ROOM_RATIO - 1.0) < 1e-6

random.seed(42)
np.random.seed(42)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load Google background recordings
google_noise = []

for filename in os.listdir(BACKGROUND_DIR):
    if filename.endswith(".wav"):
        audio, sr = librosa.load(
            os.path.join(BACKGROUND_DIR, filename),
            sr=TARGET_SR
        )
        if len(audio) >= CLIP_LENGTH:
            google_noise.append(audio)

print("Google recordings:", len(google_noise))

# Load your recordings
room_noise = []
if os.path.exists(MY_SILENCE_DIR):
    for filename in os.listdir(MY_SILENCE_DIR):
        if filename.lower().endswith((".wav", ".m4a", ".mp3", ".flac")):
            audio, sr = librosa.load(os.path.join(MY_SILENCE_DIR, filename), sr=TARGET_SR)
            if len(audio) >= CLIP_LENGTH:
                room_noise.append(audio)

print("Room recordings:", len(room_noise))

# Generate silence clips
for i in tqdm(range(NUM_SAMPLES)):
    r = random.random()

    # Pure silence
    if r < PURE_SILENCE_RATIO:
        clip = np.zeros(CLIP_LENGTH, dtype=np.float32)

    # Room recordings
    elif r < PURE_SILENCE_RATIO + ROOM_RATIO and len(room_noise) > 0:
        audio = random.choice(room_noise)
        start = random.randint(0, len(audio) - CLIP_LENGTH)
        clip = audio[start:start+CLIP_LENGTH].copy()

    # Google background recordings
    else:
        audio = random.choice(google_noise)
        start = random.randint(0, len(audio) - CLIP_LENGTH)
        clip = audio[start:start+CLIP_LENGTH].copy()
        gain = random.uniform(MIN_GAIN, MAX_GAIN)
        clip *= gain

    # Small microphone noise
    clip += np.random.normal(0, 0.0015, CLIP_LENGTH)
    clip = np.clip(clip, -1.0, 1.0)

    sf.write(os.path.join(OUTPUT_DIR, f"silence_{i:05d}.wav"), clip, TARGET_SR)

print(f"\nGenerated {NUM_SAMPLES} silence clips.")

Google recordings: 6


/tmp/ipykernel_255/3860356013.py:44: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(os.path.join(MY_SILENCE_DIR, filename), sr=TARGET_SR)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_255/3860356013.py:44: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(os.path.join(MY_SILENCE_DIR, filename), sr=TARGET_SR)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_255/3860356013.py:44: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr 

Room recordings: 4


100%|██████████| 4000/4000 [00:14<00:00, 275.27it/s]


Generated 4000 silence clips.


##List Dataset Classes and Sample Counts

In [4]:
commands = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])

print("=" * 55)
print("Speech Commands Dataset")
print("=" * 55)
print(f"{'No.':<5}{'Class':<25}{'Samples':>10}")
print("-" * 55)

total_samples = 0

for i, command in enumerate(commands, start=1):
    num_samples = len(list((data_dir / command).glob("*.wav")))
    total_samples += num_samples
    print(f"{i:<5}{command:<25}{num_samples:>10}")

print("-" * 55)
print(f"{'Total Classes':<30}: {len(commands)}")
print(f"{'Total Samples':<30}: {total_samples}")

Speech Commands Dataset
No.  Class                       Samples
-------------------------------------------------------
1    _background_noise_                6
2    backward                       1664
3    bed                            2014
4    bird                           2064
5    cat                            2031
6    dog                            2128
7    down                           3917
8    eight                          3787
9    five                           4052
10   follow                         1579
11   forward                        1557
12   four                           3728
13   go                             3880
14   happy                          2054
15   house                          2113
16   learn                          1575
17   left                           3801
18   marvin                         2100
19   nine                           3934
20   no                             3941
21   off                            3745
22   on           

## Prepare Dataset and Assign Class Labels

In [5]:
# Classes: yes = 0 no = 1 unknown = 2 Silence = 3

LIMIT = 4000

files = []
labels = []

# YES
yes_files = list((data_dir / "yes").glob("*.wav"))
random.shuffle(yes_files)
yes_files = yes_files[:LIMIT]

for f in yes_files:
    files.append(str(f))
    labels.append(0)

# NO
no_files = list((data_dir / "no").glob("*.wav"))
random.shuffle(no_files)
no_files = no_files[:LIMIT]

for f in no_files:
    files.append(str(f))
    labels.append(1)

unknown_files = []
unknown_words = ["backward","bed","bird","cat","dog","down","eight","five",
                 "follow","forward","four","go","happy","house","learn","left",
                 "marvin","nine", "off","on","one","right","seven","sheila",
                 "six","stop","three","tree","two","up","visual","wow","zero"]

for word in unknown_words:
    unknown_files.extend(list((data_dir / word).glob("*.wav")))

random.shuffle(unknown_files)

for f in unknown_files[:LIMIT]:
    files.append(str(f))
    labels.append(2)

#SILENCE
silence_files = list((data_dir / "silence").glob("*.wav"))
random.shuffle(silence_files)
silence_files = silence_files[:LIMIT]

for f in silence_files:
    files.append(str(f))
    labels.append(3)

print("Total files:", len(files))
print("YES:", labels.count(0))
print("NO:", labels.count(1))
print("UNKNOWN:", labels.count(2))
print("SILENCE:", labels.count(3))

Total files: 15941
YES: 4000
NO: 3941
UNKNOWN: 4000
SILENCE: 4000


## Audio Preprocessing and MFCC Feature Extraction

In [6]:
# WAV FILE → PCM
def audio_to_pcm(filename):
    SR = 16000

    with wave.open(filename, "rb") as wav:
        sr = wav.getframerate()
        channels = wav.getnchannels()
        width = wav.getsampwidth()
        raw = wav.readframes(wav.getnframes())

    if sr != SR:
        raise ValueError(f"Expected 16 kHz, got {sr} Hz")

    if channels != 1:
        raise ValueError("Expected mono audio")

    if width != 2:
        raise ValueError("Expected 16-bit PCM audio")

    # Convert WAV bytes → int16 PCM
    pcm = np.frombuffer(raw, dtype=np.int16)

    # Make exactly 1 second
    if len(pcm) < SR:
        pcm = np.pad(pcm, (0, SR - len(pcm)))
    else:
        pcm = pcm[:SR]

    return pcm


# PCM → MFCC
def pcm_to_mfcc(pcm):

    SR = 16000

    N_MFCC = 13
    N_MELS = 26

    FFT_SIZE = 512
    FRAME_LENGTH = 400
    FRAME_STEP = 320

    LOW_FREQ = 20
    HIGH_FREQ = 4000

    # Convert int16 PCM → float
    audio = pcm.astype(np.float32) / 32768.0

    # Pre-emphasis
    emphasized = np.empty_like(audio)
    emphasized[0] = audio[0]
    emphasized[1:] = (audio[1:] - 0.97 * audio[:-1])

    # Hann window
    window = (0.5 - 0.5 * np.cos(2 * np.pi * np.arange(FRAME_LENGTH)/(FRAME_LENGTH - 1)))

    # Mel filter bank
    def hz_to_mel(hz):
        return 2595.0 * np.log10(1.0 + hz / 700.0)

    def mel_to_hz(mel):
        return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

    mel_min = hz_to_mel(LOW_FREQ)
    mel_max = hz_to_mel(HIGH_FREQ)
    mel_points = np.linspace(mel_min, mel_max, N_MELS + 2)
    hz_points = mel_to_hz(mel_points)
    bins = np.floor((FFT_SIZE + 1)* hz_points/ SR).astype(int)

    filterbank = np.zeros((N_MELS, FFT_SIZE // 2 + 1), dtype=np.float32)

    for m in range(1, N_MELS + 1):
        left = bins[m - 1]
        center = bins[m]
        right = bins[m + 1]

        for k in range(left, center):
            if center > left:
                filterbank[m - 1, k] = ((k - left)/(center - left))

        for k in range(center, right):
            if right > center:
                filterbank[m - 1, k] = ((right - k) / (right - center))

    # DCT matrix
    dct = np.zeros((N_MFCC, N_MELS), dtype=np.float32)
    for k in range(N_MFCC):
        for n in range(N_MELS):
            dct[k, n] = np.cos(np.pi * k * (2 * n + 1)/ (2 * N_MELS))


    # Frame processing
    num_frames = 1 + ((len(emphasized) - FRAME_LENGTH)// FRAME_STEP)
    mfcc = np.zeros((N_MFCC, num_frames), dtype=np.float32)

    for i in range(num_frames):
        start = i * FRAME_STEP
        end = start + FRAME_LENGTH
        frame = emphasized[start:end]
        # Window
        frame = frame * window
        # Zero padding
        fft_input = np.zeros(FFT_SIZE,dtype=np.float32)
        fft_input[:FRAME_LENGTH] = frame
        # FFT
        spectrum = np.fft.rfft(fft_input)
        # Power spectrum
        power = (np.abs(spectrum) ** 2) / FFT_SIZE

        # Mel filter bank
        mel_energy = np.dot(filterbank, power)

        # Avoid log(0)
        mel_energy = np.maximum(mel_energy, 1e-10)

        # Log
        log_mel = np.log(mel_energy)

        # DCT
        mfcc[:, i] = np.dot(dct, log_mel)

    # Force 49 frames
    if mfcc.shape[1] < 49:
        mfcc = np.pad(mfcc,((0, 0),(0, 49 - mfcc.shape[1])))

    elif mfcc.shape[1] > 49:
        mfcc = mfcc[:, :49]

    return mfcc.astype(np.float32)

def extract_features(filename):
    pcm = audio_to_pcm(filename)
    mfcc = pcm_to_mfcc(pcm)
    return mfcc

## Extract MFCC Features and Prepare the Dataset

In [7]:
X_float32 = np.array([extract_features(f) for f in files], dtype=np.float32)
X = X_float32[..., np.newaxis]          # Shape: (N, 13, 49, 1)

y = np.array(labels, dtype=np.int32)

print("Float X shape :", X.shape)
print(np.unique(y, return_counts=True))

Float X shape : (15941, 13, 49, 1)
(array([0, 1, 2, 3], dtype=int32), array([4000, 3941, 4000, 4000]))


## Split the Dataset and Build the CNN Model

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(13,49,1)),

    tf.keras.layers.Conv2D(32, (4,10), strides=(2,3), padding="same", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(48, (3,3), strides=(2,2), padding="same", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(64, (3,3), strides=(2,2), padding="same", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(4, activation="softmax")
])


optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 7, 17, 32)      │         1,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 7, 17, 32)      │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 7, 17, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 4, 9, 48)       │        13,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 4, 9, 48)       │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 4, 9, 48)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 2, 5, 64)       │        27,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 2, 5, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 2, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,540 (177.89 KB)

 Trainable params: 45,252 (176.77 KB)

 Non-trainable params: 288 (1.12 KB)

## Train and Evaluate the CNN Model

In [9]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience= 5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)

loss, acc = model.evaluate(X_test,y_test)

print("Accuracy:", acc)

Epoch 1/50
314/314 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.7492 - loss: 0.6287 - val_accuracy: 0.8593 - val_loss: 0.4037
Epoch 2/50
314/314 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.8993 - loss: 0.2900 - val_accuracy: 0.9023 - val_loss: 0.2447
Epoch 3/50
314/314 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.9207 - loss: 0.2190 - val_accuracy: 0.9167 - val_loss: 0.2254
Epoch 4/50
314/314 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.9384 - loss: 0.1770 - val_accuracy: 0.9453 - val_loss: 0.1468
Epoch 5/50
314/314 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9442 - loss: 0.1589 - val_accuracy: 0.9346 - val_loss: 0.1764
Epoch 6/50
314/314 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9509 - loss: 0.1362 - val_accuracy: 0.9176 - val_loss: 0.2161
Epoch 7/50
314/314 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.9587 - loss: 0.1161 - val_accuracy: 0.9256 - val_loss: 0.1885
Epoch 8/50
314/314 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.9632 - loss: 0.1042 - val_acc

## Evaluate Model Performance Using Classification Metrics

In [10]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

pred = model.predict(X_test, verbose=0)

y_pred = np.argmax(pred, axis=1)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test,y_pred,target_names=["YES","NO","UNKNOWN","SILENCE"]))

Accuracy: 0.9496132134643529

Confusion Matrix
[[1144    8   40    8]
 [  13 1119   46    5]
 [  36   72 1079   13]
 [   0    0    0 1200]]

Classification Report
              precision    recall  f1-score   support

         YES       0.96      0.95      0.96      1200
          NO       0.93      0.95      0.94      1183
     UNKNOWN       0.93      0.90      0.91      1200
     SILENCE       0.98      1.00      0.99      1200

    accuracy                           0.95      4783
   macro avg       0.95      0.95      0.95      4783
weighted avg       0.95      0.95      0.95      4783



## Convert the Trained Model to TensorFlow Lite

In [11]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved.")

Saved artifact at '/tmp/tmpp9zm5hj6'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 13, 49, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  133222141576592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141578704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141579088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141584080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141576208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141584848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141582352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141586000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141576016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141577360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141575824:

## Quantize the TensorFlow Lite Model (INT8)

In [12]:
def representative_dataset():
    for i in range(300):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_int8 = converter.convert()

with open("quantized_model.tflite", "wb") as f:
    f.write(tflite_model_int8)

Saved artifact at '/tmp/tmp92kb2m1z'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 13, 49, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  133222141576592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141578704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141579088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141584080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141576208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141584848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141582352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141586000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141576016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141577360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133222141575824:

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


## Compare TensorFlow Lite Model Sizes, FLOAT32 and INT8 Model

In [13]:
size = os.path.getsize("model.tflite")
print(f"FLOAT32 Model size: {size/1024:.2f} KB")

size = os.path.getsize("quantized_model.tflite")
print(f"INT8 Model size: {size/1024:.2f} KB")

FLOAT32 Model size: 179.80 KB
INT8 Model size: 52.79 KB


## Convert the Quantized Model to a C Source File

In [14]:
!xxd -i quantized_model.tflite > model_data.cc

## Inspect the Quantized TensorFlow Lite Model

In [15]:
interpreter = tf.lite.Interpreter(model_path="quantized_model.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()

print("Shape:", input_details[0]["shape"])
print("Scale:", input_details[0]["quantization"][0])
print("Zero Point:", input_details[0]["quantization"][1])

Shape: [ 1 13 49  1]
Scale: 2.6467630863189697
Zero Point: 98


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


# Generate Embedded Test Dataset for Model Validation in Micro Controller

In [16]:
import os
import random
import numpy as np

# Configuration
NUM_PER_CLASS = 25         # 25 x 4 = 100 samples

PCM_SIZE = 16000
MFCC_SIZE = 13 * 49

random.seed(42)

# Split filenames exactly like the dataset
files = np.array(files)
labels = np.array(labels)

train_idx, test_idx = train_test_split(
    np.arange(len(files)),
    test_size=0.30,
    random_state=42,
    stratify=labels
)

test_files = files[test_idx]
test_labels = labels[test_idx]

# Select balanced samples
selected_pcm = []
selected_mfcc = []
selected_labels = []

for cls in range(4):

    idx = np.where(test_labels == cls)[0]
    idx = idx[:NUM_PER_CLASS]

    for i in idx:

        filename = test_files[i]

        pcm = audio_to_pcm(filename)
        mfcc = pcm_to_mfcc(pcm)

        selected_pcm.append(pcm.astype(np.int16))
        selected_mfcc.append(mfcc.flatten().astype(np.float32))
        selected_labels.append(cls)

selected_pcm = np.array(selected_pcm, dtype=np.int16)
selected_mfcc = np.array(selected_mfcc, dtype=np.float32)
selected_labels = np.array(selected_labels, dtype=np.uint8)

print("PCM:", selected_pcm.shape)
print("MFCC:", selected_mfcc.shape)
print("Labels:", selected_labels.shape)

# Write Header
with open("test_data.h", "w") as f:

    f.write("#ifndef TEST_DATA_H\n")
    f.write("#define TEST_DATA_H\n\n")

    f.write("#include <stdint.h>\n\n")

    f.write("#ifdef __cplusplus\n")
    f.write('extern "C" {\n')
    f.write("#endif\n\n")

    f.write(f"#define NUM_TEST_SAMPLES {len(selected_pcm)}\n")
    f.write(f"#define PCM_SIZE {PCM_SIZE}\n")
    f.write(f"#define MFCC_SIZE {MFCC_SIZE}\n\n")

    f.write("extern const int16_t test_pcm[NUM_TEST_SAMPLES][PCM_SIZE];\n")
    f.write("extern const float expected_mfcc[NUM_TEST_SAMPLES][MFCC_SIZE];\n")
    f.write("extern const uint8_t test_labels[NUM_TEST_SAMPLES];\n\n")

    f.write("#ifdef __cplusplus\n")
    f.write("}\n")
    f.write("#endif\n\n")

    f.write("#endif\n")

# Write Source
with open("test_data.c", "w") as f:

    f.write('#include "test_data.h"\n\n')

    # PCM
    f.write("const int16_t test_pcm[NUM_TEST_SAMPLES][PCM_SIZE] = {\n")

    for pcm in selected_pcm:
        f.write("    {")
        f.write(", ".join(map(str, pcm)))
        f.write("},\n")

    f.write("};\n\n")

    # Expected MFCC
    f.write("const float expected_mfcc[NUM_TEST_SAMPLES][MFCC_SIZE] = {\n")

    for mfcc in selected_mfcc:
        f.write("    {")
        f.write(", ".join(f"{x:.8f}f" for x in mfcc))
        f.write("},\n")

    f.write("};\n\n")

    # Labels
    f.write("const uint8_t test_labels[NUM_TEST_SAMPLES] = {\n    ")

    for i, label in enumerate(selected_labels):

        f.write(str(int(label)))

        if i != len(selected_labels) - 1:
            f.write(", ")

        if (i + 1) % 20 == 0:
            f.write("\n    ")

    f.write("\n};\n")

print("Generated:")
print("  test_data.h")
print("  test_data.c")
print(f"  Samples : {len(selected_pcm)}")

PCM: (100, 16000)
MFCC: (100, 637)
Labels: (100,)
Generated:
  test_data.h
  test_data.c
  Samples : 100
